# Post-Processing Ablation Study — OF_full_run_v2

This notebook evaluates the impact of **7 post-processing methods** on the
Onsets & Frames baseline model (OF_full_run_v2, 1000 epochs, full MAESTRO).

## Experimental Design

**Phase 1 — Individual isolation tests (one method at a time):**
- Exp 0: Baseline (no post-processing)
- Exp 1: Method 3 only — Minimum Note Duration
- Exp 2: Method 1 only — Onset-Conditioned Offset
- Exp 3: Method 2 only — Frame-Level Smoothing
- Exp 4: Method 4 only — Duplicate Removal
- Exp 5: Method 5 only — Chord-Aware Onset Grouping
- Exp 6: Method 6 only — Adaptive Thresholding
- Exp 7: Method 7 only — Sustain Pedal Extension

**Phase 2 — Incremental combinations (building best pipeline):**
- Exp 8: Methods 1+3 (onset offset + min duration)
- Exp 9: Methods 1+2+3 (add frame smoothing)
- Exp 10: Methods 1+2+3+4+5 (add event-level refinement)
- Exp 11: Best combo from Phase 1 winners

**Phase 3 — Final comparison and selection**

---
## Environment Setup

In [ ]:
# ── GPU check ─────────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU! Change runtime to GPU."
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [1]:
# ── Mount Google Drive ─────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted ✓")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted ✓


In [2]:
# ── Clone / update repo ────────────────────────────────────────────────────
import os
from getpass import getpass

REPO_DIR = '/content/AMT_FYP'
TOKEN    = getpass('GitHub token: ')

if not os.path.exists(REPO_DIR):
    os.system(f"git clone https://{TOKEN}@github.com/Mobinmo83/AMT_FYP.git {REPO_DIR}")
    print(f"Cloned → {REPO_DIR}")
else:
    os.system(f"cd {REPO_DIR} && git pull")
    print(f"Pulled latest → {REPO_DIR}")

GitHub token: ··········
Pulled latest → /content/AMT_FYP


In [3]:
%cd /content/AMT_FYP/piano_amt
!pip install -q -r requirements.txt
print("Packages installed ✓")

/content/AMT_FYP/piano_amt
Packages installed ✓


In [4]:
# ── sys.path + imports ─────────────────────────────────────────────────────
import sys
import mir_eval

PROJECT_ROOT = '/content/AMT_FYP/piano_amt'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.constants import N_MELS, FRAMES_PER_SECOND, N_KEYS
assert N_MELS == 229 and abs(FRAMES_PER_SECOND - 31.25) < 1e-6 and N_KEYS == 88

from models.onsets_frames.decode_advanced import advanced_rolls_to_note_events
from models.onsets_frames.evaluate_advanced import run_advanced_evaluation

print("All imports OK ✓")
print("  decode_advanced.py  ✓")
print("  evaluate_advanced.py ✓")

All imports OK ✓
  decode_advanced.py  ✓
  evaluate_advanced.py ✓


---
## Paths & Configuration

In [5]:
import os, glob

# ── Paths ────────────────────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/piano_amt'
MAESTRO_ROOT = f'{DRIVE_ROOT}/maestro-v3.0.0'
CACHE_DIR    = f'{DRIVE_ROOT}/cache'
RUNS_DIR     = f'{DRIVE_ROOT}/runs'
BEST_CKPT    = f'{RUNS_DIR}/OF_full_run_v2/checkpoints/best.pt'

# ── Evaluation settings ──────────────────────────────────────────────────────
SPLIT            = 'validation'     # 'test' for final, 'validation' for tuning
MODEL_COMPLEXITY = 48
MAX_FILES        = None       # None = all files in split

# ── Common kwargs (locked for all experiments) ───────────────────────────────
EVAL_KWARGS = dict(
    checkpoint_path      = BEST_CKPT,
    maestro_root         = MAESTRO_ROOT,
    cache_dir            = CACHE_DIR,
    split                = SPLIT,
    max_files            = MAX_FILES,
    model_complexity     = MODEL_COMPLEXITY,
    save_midi            = False,
    save_plots           = False,
    onset_threshold      = 0.5,
    frame_threshold      = 0.5,
    offset_threshold     = 0.2,
    onset_tolerance      = 0.05,
    offset_ratio         = 0.2,
    offset_min_tolerance = 0.05,
    velocity_tolerance   = 0.1,
)

# ── Storage for all results ──────────────────────────────────────────────────
ALL_SUMMARIES = {}  # config_name → summary dict

# ── Verify ───────────────────────────────────────────────────────────────────
assert os.path.exists(BEST_CKPT), f"Checkpoint not found: {BEST_CKPT}"
npzs = glob.glob(f'{CACHE_DIR}/*.npz')
print(f"Checkpoint : {BEST_CKPT}")
print(f"Cache files: {len(npzs)}")
print(f"Split      : {SPLIT}")
print(f"Max files  : {MAX_FILES or 'ALL'}")
print("\nReady ✓")

Checkpoint : /content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt
Cache files: 1276
Split      : validation
Max files  : ALL

Ready ✓


---
# PHASE 0 — Parameter Tuning using  validation set

### Exp 0 — Baseline (No Post-Processing)
All methods OFF. Should match your original `eval_test` results exactly.

In [7]:
ALL_SUMMARIES['exp0_baseline'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp0_baseline_validation',
)
print("Exp 0 DONE ✓")

Device: NVIDIA A100-SXM4-80GB
Loaded checkpoint: /content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt
  Trained for 990 epochs, val_loss=0.0453
  Model parameters: 26,491,256

  Post-processing config: 'exp0_baseline_validation'
    (no post-processing — baseline decode)

Evaluating 137/137 files from 'validation' split...
  Strategy: full-length single-pass inference + advanced post-processing

  Evaluated 137/137 files in 1990.1s

  ADVANCED EVALUATION — validation split — config: exp0_baseline_validation

  Metric                                Orig F1    Adv F1         Δ
  -----------------------------------------------------------------
  Note (onset+pitch)                     0.9270    0.9270  + 0.0000
  Note w/ offset                         0.6196    0.6196  + 0.0000
  Note w/ offset+vel                     0.5915    0.5915  + 0.0000

  Frame F1: 0.7712

  Supplementary:
    Offset MAE                 115.6 ms
    Onset MAE                  4.2 ms
    Chord 

### Exp 1 — Method 3 ONLY: Minimum Note Duration (50ms)
Increases MIN_DUR from 16ms to 50ms. Removes spurious very short notes.

In [8]:
ALL_SUMMARIES['exp1_m3_min_dur'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp1_m3_min_dur_validation',
    min_note_duration_ms=50.0,
)
print("Exp 1 DONE ✓")

Device: NVIDIA A100-SXM4-80GB
Loaded checkpoint: /content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt
  Trained for 990 epochs, val_loss=0.0453
  Model parameters: 26,491,256

  Post-processing config: 'exp1_m3_min_dur_validation'
    ✓ Min Note Duration = 50.0ms

Evaluating 137/137 files from 'validation' split...
  Strategy: full-length single-pass inference + advanced post-processing

  Evaluated 137/137 files in 1725.1s

  ADVANCED EVALUATION — validation split — config: exp1_m3_min_dur_validation

  Metric                                Orig F1    Adv F1         Δ
  -----------------------------------------------------------------
  Note (onset+pitch)                     0.9270    0.9381  + 0.0111
  Note w/ offset                         0.6196    0.6278  + 0.0082
  Note w/ offset+vel                     0.5915    0.5995  + 0.0080

  Frame F1: 0.7712

  Supplementary:
    Offset MAE                 115.6 ms
    Onset MAE                  4.2 ms
    Chord comple

### Exp 2 — Method 1 ONLY: Onset-Conditioned Offset
Uses the offset head to end notes. The offset head is trained but ignored by original decode.

In [9]:
ALL_SUMMARIES['exp2_m1_offset'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp2_m1_offset_validation',
    use_onset_conditioned_offset=True,
)
print("Exp 2 DONE ✓")

Device: NVIDIA A100-SXM4-80GB
Loaded checkpoint: /content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt
  Trained for 990 epochs, val_loss=0.0453
  Model parameters: 26,491,256

  Post-processing config: 'exp2_m1_offset_validation'
    ✓ Onset-Conditioned Offset

Evaluating 137/137 files from 'validation' split...
  Strategy: full-length single-pass inference + advanced post-processing

  Evaluated 137/137 files in 1748.1s

  ADVANCED EVALUATION — validation split — config: exp2_m1_offset_validation

  Metric                                Orig F1    Adv F1         Δ
  -----------------------------------------------------------------
  Note (onset+pitch)                     0.9270    0.9270  + 0.0000
  Note w/ offset                         0.6196    0.4688  -0.1508
  Note w/ offset+vel                     0.5915    0.4473  -0.1442

  Frame F1: 0.7712

  Supplementary:
    Offset MAE                 115.6 ms
    Onset MAE                  4.2 ms
    Chord completeness

### Exp 3 — Method 2 ONLY: Frame-Level Smoothing
Applies median filter (kernel=7) to frame roll before decoding.

In [10]:
ALL_SUMMARIES['exp3_m2_smooth'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp3_m2_smooth_validation',
    use_frame_smoothing=True,
    frame_smoothing_kernel=7,
    frame_smoothing_method='median',
)
print("Exp 3 DONE ✓")

Device: NVIDIA A100-SXM4-80GB
Loaded checkpoint: /content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt
  Trained for 990 epochs, val_loss=0.0453
  Model parameters: 26,491,256

  Post-processing config: 'exp3_m2_smooth_validation'
    ✓ Frame Smoothing (kernel=7, method=median)

Evaluating 137/137 files from 'validation' split...
  Strategy: full-length single-pass inference + advanced post-processing

  Evaluated 137/137 files in 6423.6s

  ADVANCED EVALUATION — validation split — config: exp3_m2_smooth_validation

  Metric                                Orig F1    Adv F1         Δ
  -----------------------------------------------------------------
  Note (onset+pitch)                     0.9270    0.9270  + 0.0000
  Note w/ offset                         0.6196    0.4421  -0.1775
  Note w/ offset+vel                     0.5915    0.4233  -0.1682

  Frame F1: 0.7712

  Supplementary:
    Offset MAE                 115.6 ms
    Onset MAE                  4.2 ms
    C

### Exp 4 — Method 4 ONLY: Velocity-Aware Duplicate Removal
Removes duplicate same-pitch onsets within 50ms, keeps highest velocity.

In [11]:
ALL_SUMMARIES['exp4_m4_dedup'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp4_m4_dedup_validation',
    use_duplicate_removal=True,
    duplicate_tolerance_sec=0.05,
)
print("Exp 4 DONE ✓")

Device: NVIDIA A100-SXM4-80GB
Loaded checkpoint: /content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt
  Trained for 990 epochs, val_loss=0.0453
  Model parameters: 26,491,256

  Post-processing config: 'exp4_m4_dedup_validation'
    ✓ Duplicate Removal (tol=50ms)

Evaluating 137/137 files from 'validation' split...
  Strategy: full-length single-pass inference + advanced post-processing

  Evaluated 137/137 files in 1733.3s

  ADVANCED EVALUATION — validation split — config: exp4_m4_dedup_validation

  Metric                                Orig F1    Adv F1         Δ
  -----------------------------------------------------------------
  Note (onset+pitch)                     0.9270    0.9404  + 0.0134
  Note w/ offset                         0.6196    0.6166  -0.0030
  Note w/ offset+vel                     0.5915    0.5890  -0.0025

  Frame F1: 0.7712

  Supplementary:
    Offset MAE                 115.6 ms
    Onset MAE                  4.2 ms
    Chord completene

### Exp 5 — Method 5 ONLY: Chord-Aware Onset Grouping
Snaps near-simultaneous onsets (within 30ms) to median time.

In [12]:
ALL_SUMMARIES['exp5_m5_chord'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp5_m5_chord_validation',
    use_chord_grouping=True,
    chord_tolerance_sec=0.03,
    chord_snap_to='median',
)
print("Exp 5 DONE ✓")

Device: NVIDIA A100-SXM4-80GB
Loaded checkpoint: /content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt
  Trained for 990 epochs, val_loss=0.0453
  Model parameters: 26,491,256

  Post-processing config: 'exp5_m5_chord_validation'
    ✓ Chord Grouping (tol=30ms, snap=median)

Evaluating 137/137 files from 'validation' split...
  Strategy: full-length single-pass inference + advanced post-processing

  Evaluated 137/137 files in 1741.0s

  ADVANCED EVALUATION — validation split — config: exp5_m5_chord_validation

  Metric                                Orig F1    Adv F1         Δ
  -----------------------------------------------------------------
  Note (onset+pitch)                     0.9270    0.9270  + 0.0000
  Note w/ offset                         0.6196    0.6196  + 0.0000
  Note w/ offset+vel                     0.5915    0.5915  + 0.0000

  Frame F1: 0.7712

  Supplementary:
    Offset MAE                 115.6 ms
    Onset MAE                  4.2 ms
    Chor

### Exp 6 — Method 6 ONLY: Adaptive Thresholding
Per-piece threshold adaptation: threshold = 0.5 + k × std(activations).

In [6]:
ALL_SUMMARIES['exp6_m6_adaptive'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp6_m6_adaptive_validation',
    use_adaptive_thresholds=True,
    adaptive_onset_k=0.5,
    adaptive_frame_k=0.5,
)
print("Exp 6 DONE ✓")

Device: NVIDIA A100-SXM4-80GB
Loaded checkpoint: /content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt
  Trained for 990 epochs, val_loss=0.0453
  Model parameters: 26,491,256

  Post-processing config: 'exp6_m6_adaptive_validation'
    ✓ Adaptive Thresholds (onset_k=0.5, frame_k=0.5)

Evaluating 137/137 files from 'validation' split...
  Strategy: full-length single-pass inference + advanced post-processing

  Evaluated 137/137 files in 1752.8s

  ADVANCED EVALUATION — validation split — config: exp6_m6_adaptive_validation

  Metric                                Orig F1    Adv F1         Δ
  -----------------------------------------------------------------
  Note (onset+pitch)                     0.9270    0.9255  -0.0015
  Note w/ offset                         0.6196    0.6177  -0.0019
  Note w/ offset+vel                     0.5915    0.5898  -0.0017

  Frame F1: 0.7712

  Supplementary:
    Offset MAE                 115.6 ms
    Onset MAE                  4.2 

### Exp 7 — Method 7 ONLY: Sustain Pedal Extension
Extends note offsets in regions with high simultaneous activity (likely pedal).

In [7]:
ALL_SUMMARIES['exp7_m7_pedal'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp7_m7_pedal_validation',
    use_pedal_extension=True,
    pedal_energy_threshold=10.0,
    pedal_max_extension_sec=2.0,
)
print("Exp 7 DONE ✓")

Device: NVIDIA A100-SXM4-80GB
Loaded checkpoint: /content/drive/MyDrive/piano_amt/runs/OF_full_run_v2/checkpoints/best.pt
  Trained for 990 epochs, val_loss=0.0453
  Model parameters: 26,491,256

  Post-processing config: 'exp7_m7_pedal_validation'
    ✓ Pedal Extension (threshold=10.0, max=2.0s)

Evaluating 137/137 files from 'validation' split...
  Strategy: full-length single-pass inference + advanced post-processing

  Evaluated 137/137 files in 1769.1s

  ADVANCED EVALUATION — validation split — config: exp7_m7_pedal_validation

  Metric                                Orig F1    Adv F1         Δ
  -----------------------------------------------------------------
  Note (onset+pitch)                     0.9270    0.9270  + 0.0000
  Note w/ offset                         0.6196    0.6196  + 0.0000
  Note w/ offset+vel                     0.5915    0.5915  + 0.0000

  Frame F1: 0.7712

  Supplementary:
    Offset MAE                 115.6 ms
    Onset MAE                  4.2 ms
    

---
# PHASE 1 — Individual Method Isolation Tests

Each experiment enables exactly ONE method to measure its isolated effect.

### Exp 0 — Baseline (No Post-Processing)
All methods OFF. Should match your original `eval_test` results exactly.

In [ ]:
ALL_SUMMARIES['exp0_baseline'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp0_baseline',
)
print("Exp 0 DONE ✓")

### Exp 1 — Method 3 ONLY: Minimum Note Duration (50ms)
Increases MIN_DUR from 16ms to 50ms. Removes spurious very short notes.

In [ ]:
ALL_SUMMARIES['exp1_m3_min_dur'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp1_m3_min_dur',
    min_note_duration_ms=50.0,
)
print("Exp 1 DONE ✓")

### Exp 2 — Method 1 ONLY: Onset-Conditioned Offset
Uses the offset head to end notes. The offset head is trained but ignored by original decode.

In [ ]:
ALL_SUMMARIES['exp2_m1_offset'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp2_m1_offset',
    use_onset_conditioned_offset=True,
)
print("Exp 2 DONE ✓")

### Exp 3 — Method 2 ONLY: Frame-Level Smoothing
Applies median filter (kernel=7) to frame roll before decoding.

In [ ]:
ALL_SUMMARIES['exp3_m2_smooth'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp3_m2_smooth',
    use_frame_smoothing=True,
    frame_smoothing_kernel=7,
    frame_smoothing_method='median',
)
print("Exp 3 DONE ✓")

### Exp 4 — Method 4 ONLY: Velocity-Aware Duplicate Removal
Removes duplicate same-pitch onsets within 50ms, keeps highest velocity.

In [ ]:
ALL_SUMMARIES['exp4_m4_dedup'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp4_m4_dedup',
    use_duplicate_removal=True,
    duplicate_tolerance_sec=0.05,
)
print("Exp 4 DONE ✓")

### Exp 5 — Method 5 ONLY: Chord-Aware Onset Grouping
Snaps near-simultaneous onsets (within 30ms) to median time.

In [ ]:
ALL_SUMMARIES['exp5_m5_chord'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp5_m5_chord',
    use_chord_grouping=True,
    chord_tolerance_sec=0.03,
    chord_snap_to='median',
)
print("Exp 5 DONE ✓")

### Exp 6 — Method 6 ONLY: Adaptive Thresholding
Per-piece threshold adaptation: threshold = 0.5 + k × std(activations).

In [ ]:
ALL_SUMMARIES['exp6_m6_adaptive'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp6_m6_adaptive',
    use_adaptive_thresholds=True,
    adaptive_onset_k=0.5,
    adaptive_frame_k=0.5,
)
print("Exp 6 DONE ✓")

### Exp 7 — Method 7 ONLY: Sustain Pedal Extension
Extends note offsets in regions with high simultaneous activity (likely pedal).

In [ ]:
ALL_SUMMARIES['exp7_m7_pedal'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp7_m7_pedal',
    use_pedal_extension=True,
    pedal_energy_threshold=10.0,
    pedal_max_extension_sec=2.0,
)
print("Exp 7 DONE ✓")

---
## Phase 1 Results — Individual Method Comparison

In [ ]:
# ── Phase 1 Results Table ────────────────────────────────────────────────────
import pandas as pd

phase1_configs = [
    'exp0_baseline', 'exp1_m3_min_dur', 'exp2_m1_offset', 'exp3_m2_smooth',
    'exp4_m4_dedup', 'exp5_m5_chord', 'exp6_m6_adaptive', 'exp7_m7_pedal'
]

baseline = ALL_SUMMARIES.get('exp0_baseline', {})
base_note  = baseline.get('adv_note_f1', baseline.get('note_f1', 0))
base_noff  = baseline.get('adv_note_with_offset_f1', baseline.get('note_with_offset_f1', 0))
base_noffv = baseline.get('adv_note_with_offset_vel_f1', baseline.get('note_with_offset_vel_f1', 0))

print('\n' + '=' * 95)
print('PHASE 1 — INDIVIDUAL METHOD ISOLATION RESULTS')
print('=' * 95)
print(f"\nBaseline: Note F1={base_note:.4f}  N+Off F1={base_noff:.4f}  N+O+V F1={base_noffv:.4f}")
print(f"\n{'Config':<22s}  {'Adv Note F1':>11s}  {'Adv N+O F1':>11s}  {'Adv N+O+V':>10s}  {'Δ Note':>7s}  {'Δ N+O':>7s}  {'Δ N+O+V':>8s}  {'Dup Rate':>8s}")
print('-' * 95)

for cfg in phase1_configs:
    s = ALL_SUMMARIES.get(cfg, {})
    if not s:
        continue
    adv_n   = s.get('adv_note_f1', s.get('note_f1', 0))
    adv_no  = s.get('adv_note_with_offset_f1', s.get('note_with_offset_f1', 0))
    adv_nov = s.get('adv_note_with_offset_vel_f1', s.get('note_with_offset_vel_f1', 0))
    dn  = adv_n - base_note
    dno = adv_no - base_noff
    dnov = adv_nov - base_noffv
    dup = s.get('ea_duplicate_note_rate', 0)
    sn  = '+' if dn >= 0 else ''
    sno = '+' if dno >= 0 else ''
    snov = '+' if dnov >= 0 else ''
    print(f"{cfg:<22s}  {adv_n:>11.4f}  {adv_no:>11.4f}  {adv_nov:>10.4f}  {sn}{dn:>6.4f}  {sno}{dno:>6.4f}  {snov}{dnov:>7.4f}  {dup:>8.4f}")

print()

---
# PHASE 2 — Combination Experiments

Based on Phase 1 results, combine methods that individually showed improvement.

### Exp 8 — Methods 1+3: Onset-Conditioned Offset + Min Duration

In [ ]:
ALL_SUMMARIES['exp8_m1_m3'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp8_m1_m3',
    use_onset_conditioned_offset=True,
    min_note_duration_ms=50.0,
)
print("Exp 8 DONE ✓")

### Exp 9 — Methods 1+2+3: Add Frame Smoothing

In [ ]:
ALL_SUMMARIES['exp9_m1_m2_m3'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp9_m1_m2_m3',
    use_onset_conditioned_offset=True,
    use_frame_smoothing=True,
    frame_smoothing_kernel=7,
    frame_smoothing_method='median',
    min_note_duration_ms=50.0,
)
print("Exp 9 DONE ✓")

### Exp 10 — Methods 1+2+3+4+5: Add Event Refinement

In [ ]:
ALL_SUMMARIES['exp10_m1_to_m5'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp10_m1_to_m5',
    use_onset_conditioned_offset=True,
    use_frame_smoothing=True,
    frame_smoothing_kernel=7,
    frame_smoothing_method='median',
    min_note_duration_ms=50.0,
    use_duplicate_removal=True,
    duplicate_tolerance_sec=0.05,
    use_chord_grouping=True,
    chord_tolerance_sec=0.03,
    chord_snap_to='median',
)
print("Exp 10 DONE ✓")

### Exp 11 — All Methods (1–7)

In [ ]:
ALL_SUMMARIES['exp11_all'] = run_advanced_evaluation(
    **EVAL_KWARGS,
    config_name='exp11_all',
    use_onset_conditioned_offset=True,
    use_frame_smoothing=True,
    frame_smoothing_kernel=7,
    frame_smoothing_method='median',
    min_note_duration_ms=50.0,
    use_duplicate_removal=True,
    duplicate_tolerance_sec=0.05,
    use_chord_grouping=True,
    chord_tolerance_sec=0.03,
    chord_snap_to='median',
    use_adaptive_thresholds=True,
    adaptive_onset_k=0.5,
    adaptive_frame_k=0.5,
    use_pedal_extension=True,
    pedal_energy_threshold=10.0,
    pedal_max_extension_sec=2.0,
)
print("Exp 11 DONE ✓")

---
# PHASE 3 — Final Comparison & Selection

In [ ]:
# ── Complete Results Table ───────────────────────────────────────────────────
import json
from pathlib import Path

run_dir = Path(RUNS_DIR) / 'OF_full_run_v2'

# Also load from disk in case notebook was restarted
eval_dirs = sorted(run_dir.glob(f'eval_{SPLIT}*'))
for d in eval_dirs:
    sf = d / 'summary_metrics.json'
    if sf.exists():
        cfg_name = d.name.replace(f'eval_{SPLIT}_', '')
        if cfg_name == f'eval_{SPLIT}':
            cfg_name = 'original'
        if cfg_name not in ALL_SUMMARIES:
            with open(sf) as f:
                ALL_SUMMARIES[cfg_name] = json.load(f)

print(f"Total configs loaded: {len(ALL_SUMMARIES)}")
for k in sorted(ALL_SUMMARIES.keys()):
    print(f"  {k}")

In [ ]:
# ── Full Comparison Table ────────────────────────────────────────────────────
import pandas as pd

baseline = ALL_SUMMARIES.get('exp0_baseline', {})
base_note  = baseline.get('adv_note_f1', baseline.get('note_f1', 0))
base_noff  = baseline.get('adv_note_with_offset_f1', baseline.get('note_with_offset_f1', 0))
base_noffv = baseline.get('adv_note_with_offset_vel_f1', baseline.get('note_with_offset_vel_f1', 0))

# Sort: phase 1 first, then phase 2
ordered = sorted(ALL_SUMMARIES.keys())

print('\n' + '=' * 110)
print('COMPLETE POST-PROCESSING ABLATION RESULTS')
print('=' * 110)
print(f"\n{'Config':<22s}  {'Adv Note':>8s}  {'Adv N+O':>8s}  {'Adv NOV':>8s}  {'Frame':>7s}  {'Δ Note':>7s}  {'Δ N+O':>7s}  {'Δ NOV':>7s}  {'OffMAE':>7s}  {'ChordC':>7s}  {'DupR':>6s}")
print('-' * 110)

for cfg in ordered:
    s = ALL_SUMMARIES[cfg]
    adv_n   = s.get('adv_note_f1', s.get('note_f1', 0))
    adv_no  = s.get('adv_note_with_offset_f1', s.get('note_with_offset_f1', 0))
    adv_nov = s.get('adv_note_with_offset_vel_f1', s.get('note_with_offset_vel_f1', 0))
    frame   = s.get('frame_f1', 0)
    off_mae = s.get('ea_offset_mae_ms', 0)
    chord_c = s.get('ea_chord_completeness', 0)
    dup     = s.get('ea_duplicate_note_rate', 0)
    dn   = adv_n - base_note
    dno  = adv_no - base_noff
    dnov = adv_nov - base_noffv
    sn  = '+' if dn >= 0 else ''
    sno = '+' if dno >= 0 else ''
    snov = '+' if dnov >= 0 else ''
    print(f"{cfg:<22s}  {adv_n:>8.4f}  {adv_no:>8.4f}  {adv_nov:>8.4f}  {frame:>7.4f}  {sn}{dn:>6.4f}  {sno}{dno:>6.4f}  {snov}{dnov:>6.4f}  {off_mae:>7.1f}  {chord_c:>7.4f}  {dup:>6.4f}")

print()

In [ ]:
# ── Bar Chart ────────────────────────────────────────────────────────────────
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image

ordered = sorted(ALL_SUMMARIES.keys())
configs = ordered
n_configs = len(configs)

metrics_to_plot = [
    ('adv_note_f1', 'note_f1', 'Note F1'),
    ('adv_note_with_offset_f1', 'note_with_offset_f1', 'Note+Off F1'),
    ('adv_note_with_offset_vel_f1', 'note_with_offset_vel_f1', 'N+O+V F1'),
]

x = np.arange(len(metrics_to_plot))
width = 0.8 / max(n_configs, 1)
colors = plt.cm.tab20(np.linspace(0, 1, max(n_configs, 1)))

fig, ax = plt.subplots(figsize=(16, 7))

for i, cfg in enumerate(configs):
    s = ALL_SUMMARIES[cfg]
    vals = []
    for adv_key, orig_key, _ in metrics_to_plot:
        vals.append(s.get(adv_key, s.get(orig_key, 0)))
    offset = (i - n_configs / 2 + 0.5) * width
    bars = ax.bar(x + offset, vals, width, label=cfg, color=colors[i], alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.003,
                f"{v:.3f}", ha='center', va='bottom', fontsize=6, rotation=60)

ax.set_xticks(x)
ax.set_xticklabels([label for _, _, label in metrics_to_plot], fontsize=12)
ax.set_ylim(0, 1.08)
ax.set_ylabel('F1 Score', fontsize=12)
ax.set_title(f'Post-Processing Ablation — OF_full_run_v2 ({SPLIT} split)', fontsize=14)
ax.legend(fontsize=6, loc='lower right', ncol=3)
ax.grid(True, axis='y', alpha=0.3)

plt.tight_layout()
save_path = str(run_dir / f'pp_ablation_{SPLIT}.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
print(f"Chart saved → {save_path}")
plt.close(fig)
Image(save_path)

In [ ]:
# ── Find Best Config ────────────────────────────────────────────────────────

best_cfg = None
best_noff = 0

for cfg, s in ALL_SUMMARIES.items():
    noff = s.get('adv_note_with_offset_f1', s.get('note_with_offset_f1', 0))
    if noff > best_noff:
        best_noff = noff
        best_cfg = cfg

print('=' * 60)
print('BEST POST-PROCESSING CONFIGURATION')
print('=' * 60)

if best_cfg:
    s = ALL_SUMMARIES[best_cfg]
    print(f"\n  Config: {best_cfg}")
    print(f"  Note F1:       {s.get('adv_note_f1', s.get('note_f1', 0)):.4f}  (Δ {s.get('adv_note_f1', s.get('note_f1', 0)) - base_note:+.4f})")
    print(f"  Note+Off F1:   {best_noff:.4f}  (Δ {best_noff - base_noff:+.4f})")
    adv_nov = s.get('adv_note_with_offset_vel_f1', s.get('note_with_offset_vel_f1', 0))
    print(f"  N+O+V F1:      {adv_nov:.4f}  (Δ {adv_nov - base_noffv:+.4f})")

    pp = s.get('post_processing', {})
    if pp:
        print(f"\n  Active methods:")
        if pp.get('use_onset_conditioned_offset'): print(f"    ✓ M1: Onset-Conditioned Offset")
        if pp.get('use_frame_smoothing'): print(f"    ✓ M2: Frame Smoothing (kernel={pp.get('frame_smoothing_kernel')})")
        if pp.get('min_note_duration_ms', 16) != 16: print(f"    ✓ M3: Min Duration = {pp.get('min_note_duration_ms')}ms")
        if pp.get('use_duplicate_removal'): print(f"    ✓ M4: Duplicate Removal")
        if pp.get('use_chord_grouping'): print(f"    ✓ M5: Chord Grouping")
        if pp.get('use_adaptive_thresholds'): print(f"    ✓ M6: Adaptive Thresholds")
        if pp.get('use_pedal_extension'): print(f"    ✓ M7: Pedal Extension")

    print(f"\n  → Use this config as input to your Transformer refiner.")

print('=' * 60)

In [ ]:
# ── Save CSV for Dissertation ────────────────────────────────────────────────

rows_csv = []
for cfg in sorted(ALL_SUMMARIES.keys()):
    s = ALL_SUMMARIES[cfg]
    rows_csv.append({
        'Config': cfg,
        'Note F1': round(s.get('adv_note_f1', s.get('note_f1', 0)), 4),
        'Note+Off F1': round(s.get('adv_note_with_offset_f1', s.get('note_with_offset_f1', 0)), 4),
        'N+O+V F1': round(s.get('adv_note_with_offset_vel_f1', s.get('note_with_offset_vel_f1', 0)), 4),
        'Frame F1': round(s.get('frame_f1', 0), 4),
        'Offset MAE (ms)': round(s.get('ea_offset_mae_ms', 0), 1),
        'Onset MAE (ms)': round(s.get('ea_onset_mae_ms', 0), 1),
        'Chord Comp': round(s.get('ea_chord_completeness', 0), 4),
        'Dup Rate': round(s.get('ea_duplicate_note_rate', 0), 4),
        'Eval Time (s)': s.get('eval_time_total_s', 0),
    })

df = pd.DataFrame(rows_csv)
csv_path = str(run_dir / f'pp_ablation_{SPLIT}.csv')
df.to_csv(csv_path, index=False)
print(f"CSV saved → {csv_path}")
print()
print(df.to_string(index=False))